# Ch2.3 Solving Systems of Linear Equations

## Concept Summary

- A system of linear equations can be represented as an augmented matrix [A|b] and solved via Gaussian elimination
- Gaussian elimination transforms the matrix into Row Echelon Form (REF), and further into Reduced Row Echelon Form (RREF)
- Elementary row operations: swap two rows, multiply a row by a nonzero scalar, add a multiple of one row to another
- Three possible outcomes: one solution, no solution, infinitely many solutions
- Particular solution: set all free variables to 0
- General solution: particular solution + all solutions to Ax = 0
- Free variables correspond to non-pivot columns; basic variables correspond to pivot columns
- The Minus-1 Trick extends RREF to a square matrix by inserting -1 at free variable positions, making null space basis vectors directly readable from the columns
- Number of solutions can be determined by comparing rank(A) and rank([A|b]) with the number of variables n

## Problems

### Problem 1 (Unique Solution)
x1 + x2 = 3  
2x1 − x2 = 0

Augmented matrix:
[1  1 | 3]
[2 -1 | 0]

R2 − 2R1:
[1  1 | 3]
[0 -3 |-6]

From row 2: x2 = 2  
Substituting: x1 = 1

**Solution: x1 = 1, x2 = 2**


### Problem 2 (Infinitely Many Solutions)
x1 + 2x2 + x3 = 4  
2x1 + 4x2 + 2x3 = 8

R2 − 2R1:
[1  2  1 | 4]
[0  0  0 | 0]

x2, x3 are free variables (λ1, λ2)  
x1 = 4 − 2λ1 − λ2

General solution:
x = [4, 0, 0] + λ1[−2, 1, 0] + λ2[−1, 0, 1]


### Problem 3 (No Solution)
x1 + x2 = 3  
x1 + x2 = 5

R2 − R1:
[1  1 | 3]
[0  0 | 2]

Row 2 gives 0 = 2, contradiction.

**No solution.**

## Verification

In [5]:
import numpy as np

# Problem 1
A1 = np.array([[1, 1], [2, -1]], dtype=float)
b1 = np.array([3, 0], dtype=float)
print("Problem 1:", np.linalg.solve(A1, b1))

# Problem 2
A2 = np.array([[1, 2, 1], [2, 4, 2]], dtype=float)
b2 = np.array([4, 8], dtype=float)
x2, _, _, _ = np.linalg.lstsq(A2, b2, rcond=None)
print("Problem 2 (particular):", x2)
print("Verification:", A2 @ x2)

# Problem 3 
A3 = np.array([[1, 1], [1, 1]], dtype=float)
b3 = np.array([3, 5], dtype=float)
try:
    np.linalg.solve(A3, b3)
except np.linalg.LinAlgError as e:
    print("Problem 3:", e)

Problem 1: [1. 2.]
Problem 2 (particular): [0.66666667 1.33333333 0.66666667]
Verification: [4. 8.]
Problem 3: Singular matrix


## Utility Function (General Solver)

In [6]:
def solve_linear_system(A, b):
    Ab = np.column_stack([A, b])
    rank_A = np.linalg.matrix_rank(A)
    rank_Ab = np.linalg.matrix_rank(Ab)
    n = A.shape[1]

    if rank_A != rank_Ab:
        print("No solution")
        return None
    elif rank_A == n:
        print("Unique solution")
        x = np.linalg.solve(A, b)
        print("Solution:", x)
        return x
    else:
        print("Infinitely many solutions (returning particular solution)")
        x, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        print("Particular solution:", x)
        return x

solve_linear_system(A1, b1)
solve_linear_system(A2, b2)
solve_linear_system(A3, b3)

Unique solution
Solution: [1. 2.]
Infinitely many solutions (returning particular solution)
Particular solution: [0.66666667 1.33333333 0.66666667]
No solution


## Reflection Note

Gaussian elimination is straightforward on paper, but translating it
into code surfaces a question that the math itself does not raise.
"How do you choose the right solver when you do not know the answer
in advance?"

numpy offers np.linalg.solve for unique solutions and np.linalg.lstsq
for underdetermined systems, but neither function tells you which case
you are in. The natural response was to check rank(A) and rank([A|b])
against the number of variables first, then branch accordingly.
Wrapping this logic into a single function made the workflow feel
complete rather than piecemeal.

In Problem 2, the particular solution from hand calculation was
[4, 0, 0] while lstsq returned [0.667, 1.333, 0.667]. Since Ax = b
means a valid solution must satisfy A @ x = b, both were verified
directly. Both produced [4, 8], confirming that particular solutions
are not unique when infinitely many solutions exist. lstsq returns
the one with the smallest vector length among all valid solutions.